In [ ]:
# Если ultralytics ещё не установлен, раскомментируй и выполни эту ячейку.
# В Jupyter лучше использовать именно %pip, чтобы пакет поставился в текущее ядро ноутбука.

%pip install -U ultralytics

: 

In [1]:
from pathlib import Path
import sys
import pandas as pd

DATASET_ROOT = Path(r"F:\kurs_work\processed\AOD4_clean")
DATA_YAML = DATASET_ROOT / "data.yaml"

RUNS_ROOT = Path(r"F:\kurs_work\runs")

print("Python:", sys.version)
print("DATA_YAML:", DATA_YAML)
print("DATA_YAML exists:", DATA_YAML.exists())

print("\nСодержимое data.yaml:")
print(DATA_YAML.read_text(encoding="utf-8"))

Python: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]
DATA_YAML: F:\kurs_work\processed\AOD4_clean\data.yaml
DATA_YAML exists: True

Содержимое data.yaml:
path: F:/kurs_work/processed/AOD4_clean

train: images/train
val: images/valid
test: images/test

names:
  0: airplane
  1: bird
  2: drone
  3: helicopter



In [9]:

#%pip install torch 

  Using cached torch-2.12.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
Using cached torch-2.12.0-cp313-cp313-win_amd64.whl (123.0 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ultralytics 8.4.51 requires torchvision>=0.9.0, which is not installed.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = 0
else:
    print("GPU не обнаружена, обучение пойдёт на CPU. Это будет значительно медленнее.")
    DEVICE = "cpu"

PyTorch: 2.6.0+cu124
CUDA доступна: True
GPU: NVIDIA GeForce GTX 1080


In [3]:
# Быстрая проверка, что структура датасета корректная

def count_files(folder: Path, extensions=None):
    if extensions is None:
        return len([path for path in folder.iterdir() if path.is_file()])
    extensions = {ext.lower() for ext in extensions}
    return len([
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in extensions
    ])

check_rows = []

for split in ["train", "valid", "test"]:
    images_dir = DATASET_ROOT / "images" / split
    labels_dir = DATASET_ROOT / "labels" / split

    images_count = count_files(images_dir, {".jpg", ".jpeg", ".png"})
    labels_count = count_files(labels_dir, {".txt"})

    check_rows.append({
        "split": split,
        "images": images_count,
        "labels": labels_count,
        "ok": images_count == labels_count,
    })

check_df = pd.DataFrame(check_rows)
display(check_df)

assert DATA_YAML.exists(), "data.yaml не найден"
assert all(check_df["ok"]), "Количество изображений и label-файлов не совпадает"

,split,images,labels,ok
0,train,15761,15761,True
1,valid,4514,4514,True
2,test,2241,2241,True


In [1]:
import sys
import torch

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Тест CUDA
    try:
        x = torch.randn(1000, 1000).cuda()
        y = x @ x.T
        y.cpu()
        print("CUDA работает корректно")
    except Exception as e:
        print(f"CUDA не работает: {e}")

Python: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.7.1+cu118
CUDA: 11.8
CUDA available: True
GPU: NVIDIA GeForce GTX 1080
Memory: 8.6 GB
CUDA работает корректно


In [7]:
import gc
import torch
from ultralytics import YOLO

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

MODEL_NAME = "yolo11n.pt"
EXPERIMENT_NAME = "aod41_yolo11n_safe_640_10_30"

EPOCHS = 30
IMAGE_SIZE = 640
BATCH_SIZE = 10

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=1,
    project=str(RUNS_ROOT),
    name=EXPERIMENT_NAME,
    pretrained=True,
    seed=42,
    patience=10,
    cache=False,
    amp=False,
    deterministic=False,
    exist_ok=True,
)

Ultralytics 8.4.51  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=10, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\kurs_work\processed\AOD4_clean\data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aod41_yolo11n_safe_640_10_30, nbs=64, nms=False, opset=None, optimize=False, o

In [ ]:
MODEL_NAME = "yolo11n.pt"   
EXPERIMENT_NAME = "aod4_yolo11n_baseline_img640"

EPOCHS = 3
IMAGE_SIZE = 416
BATCH_SIZE = 8
MODEL_NAME = "yolo11n.pt"
model = YOLO(MODEL_NAME)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    project=str(RUNS_DIR),
    name=EXPERIMENT_NAME,
)

NameError: name 'RUNS_DIR' is not defined

In [ ]:
MODEL_NAME = "yolo11n.pt"    
EXPERIMENT_NAME = "aod4_yolo11n_baseline_img640"

EPOCHS = 5
IMAGE_SIZE = 640
BATCH_SIZE = 16

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=4,
    project=str(RUNS_ROOT),
    name=EXPERIMENT_NAME,
    pretrained=True,
    seed=42,
    patience=10,
    cache=False,

)

Ultralytics 8.4.51  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce GTX 1080, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\kurs_work\processed\AOD4_clean\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aod4_yolo11n_baseline_img640-4, nbs=64, nms=False, opset=None, optimize=False, o

RuntimeError: Caught RuntimeError in pin memory thread for device 0.
Original Traceback (most recent call last):
  File "c:\Users\Иван\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\_utils\pin_memory.py", line 41, in do_one_step
    data = pin_memory(data, device)
  File "c:\Users\Иван\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\_utils\pin_memory.py", line 75, in pin_memory
    {k: pin_memory(sample, device) for k, sample in data.items()}
        ~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "c:\Users\Иван\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\_utils\pin_memory.py", line 64, in pin_memory
    return data.pin_memory(device)
           ~~~~~~~~~~~~~~~^^^^^^^^
RuntimeError: CUDA error: resource already mapped
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CU

In [ ]:


run_dir = RUNS_ROOT / EXPERIMENT_NAME
best_model_path = run_dir / "weights" / "best.pt"
last_model_path = run_dir / "weights" / "last.pt"

print("run_dir:", run_dir)
print("best.pt exists:", best_model_path.exists())
print("last.pt exists:", last_model_path.exists())

In [ ]:

results_csv = run_dir / "results.csv"

if results_csv.exists():
    results_df = pd.read_csv(results_csv)
    display(results_df.tail())

    # Коротко выводим лучшие значения по основным метрикам, если такие колонки есть.
    for column in results_df.columns:
        if "metrics/mAP50" in column or "metrics/mAP50-95" in column:
            print(column, "max =", results_df[column].max())
else:
    print("results.csv пока не найден. Возможно, обучение ещё не завершено.")

In [ ]:

trained_model = YOLO(str(best_model_path))

test_metrics = trained_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(RUNS_ROOT),
    name=f"{EXPERIMENT_NAME}_test_eval",
)

print(test_metrics)

In [ ]:


valid_images_dir = DATASET_ROOT / "images" / "valid"

predict_results = trained_model.predict(
    source=str(valid_images_dir),
    imgsz=IMAGE_SIZE,
    conf=0.25,
    device=DEVICE,
    max_det=20,
    save=True,
    project=str(RUNS_ROOT),
    name=f"{EXPERIMENT_NAME}_predict_valid_examples",
)

print("Предсказания сохранены в:")
print(RUNS_ROOT / f"{EXPERIMENT_NAME}_predict_valid_examples")

In [ ]:
RUN_QUICK_COMPARISON = False

if RUN_QUICK_COMPARISON:
    models_to_test = ["yolov8n.pt", "yolo11n.pt"]
    quick_rows = []

    for model_name in models_to_test:
        exp_name = f"aod4_{model_name.replace('.pt', '')}_quick_img640"

        quick_model = YOLO(model_name)
        quick_model.train(
            data=str(DATA_YAML),
            epochs=15,
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            device=DEVICE,
            workers=4,
            project=str(RUNS_ROOT),
            name=exp_name,
            pretrained=True,
            seed=42,
            patience=5,
            cache=False,
        )

        quick_best = RUNS_ROOT / exp_name / "weights" / "best.pt"
        quick_eval = YOLO(str(quick_best)).val(
            data=str(DATA_YAML),
            split="test",
            imgsz=IMAGE_SIZE,
            batch=BATCH_SIZE,
            device=DEVICE,
            project=str(RUNS_ROOT),
            name=f"{exp_name}_test_eval",
        )

        quick_rows.append({
            "model": model_name,
            "experiment": exp_name,
            "best_model": str(quick_best),
        })

    quick_df = pd.DataFrame(quick_rows)
    display(quick_df)
else:
    print("RUN_QUICK_COMPARISON = False, ячейка пропущена.")